In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 1200)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
PROJECT_DIR = Path(
    r"F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "figures"

EXCEL_PATH = RAW_DIR / "BK_AUG2026.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Excel file:", EXCEL_PATH)
print("Figures directory:", OUTPUT_DIR)

Project directory: F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand
Excel file: F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand\data\raw\BK_AUG2026.xlsx
Figures directory: F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand\outputs\figures


In [3]:
if not EXCEL_PATH.exists():
    raise FileNotFoundError(
        f"Excel file not found: {EXCEL_PATH}"
    )

print("Excel file found successfully!")

Excel file found successfully!


In [4]:
excel_file = pd.ExcelFile(EXCEL_PATH)

print("Available Excel sheets:")
print(excel_file.sheet_names)

Available Excel sheets:
['1-15', '16-31']


In [ ]:
df = pd.read_excel(
    EXCEL_PATH,
    sheet_name=excel_file.sheet_names[0]
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

In [ ]:
display(df.head(10))

In [ ]:
display(df.tail(10))

In [ ]:
rows, columns = df.shape

print("Number of rows:", rows)
print("Number of columns:", columns)

In [ ]:
print("Column names:")

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

In [ ]:
print("Data types of each column:")
display(df.dtypes)

In [ ]:
df.info()

In [ ]:
display(df.describe(include="all").T)

In [ ]:
missing_values = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing_Values": missing_values.values,
    "Missing_Percentage": (
        missing_values.values / len(df) * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(missing_summary)

In [ ]:
plt.figure(figsize=(14, 6))

sns.heatmap(
    df.isnull(),
    cbar=False,
    yticklabels=False,
    cmap="viridis"
)

plt.title("Missing Values in Logistics Booking Dataset")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.xticks(rotation=90)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "missing_values_heatmap.png",
    dpi=300
)

plt.show()

In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)
print("Total rows:", len(df))

In [ ]:
duplicates = df[df.duplicated(keep=False)]

print("Duplicate records:")
display(duplicates.head(20))

In [ ]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numeric columns:")
print(numeric_columns)

In [ ]:
categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

In [ ]:
date_candidates = [
    "BKDATE",
    "DATE",
    "Date",
    "date",
    "booking_date"
]

date_col = next(
    (col for col in date_candidates if col in df.columns),
    None
)

if date_col is None:
    print("Date column not identified automatically.")
    print("Available columns:", df.columns.tolist())
else:
    print("Date column:", date_col)

In [ ]:
if date_col is not None:
    df[date_col] = pd.to_datetime(
        df[date_col],
        errors="coerce",
        dayfirst=True
    )

    print("Date column converted.")
    print("Invalid dates:", df[date_col].isna().sum())

    display(df[[date_col]].head())

In [ ]:
if date_col is not None:
    valid_dates = df[date_col].dropna()

    if not valid_dates.empty:
        print("Earliest booking date:", valid_dates.min())
        print("Latest booking date:", valid_dates.max())
        print("Number of unique booking dates:",
              valid_dates.nunique())
    else:
        print("No valid dates found.")
        

In [ ]:
if date_col is not None:
    date_data = df.dropna(subset=[date_col]).copy()

    daily_bookings = (
        date_data.groupby(date_col)
        .size()
        .sort_index()
    )

    plt.figure(figsize=(14, 5))

    plt.plot(
        daily_bookings.index,
        daily_bookings.values,
        marker="o",
        linewidth=1.5
    )

    plt.title("Daily Logistics Booking Count")
    plt.xlabel("Booking Date")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / "daily_booking_count.png",
        dpi=300
    )

    plt.show()

In [ ]:
if numeric_columns:
    df[numeric_columns].hist(
        figsize=(16, 12),
        bins=30
    )

    plt.suptitle(
        "Distribution of Numeric Features",
        fontsize=16
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / "numeric_feature_distributions.png",
        dpi=300
    )

    plt.show()
else:
    print("No numeric columns available.")

In [ ]:
for col in categorical_columns:
    print(f"\nColumn: {col}")
    print("Unique values:", df[col].nunique())
    display(df[col].value_counts(dropna=False).head(10))

In [ ]:
mode_candidates = ["BKMODE", "booking_mode", "Booking_Mode"]

mode_col = next(
    (col for col in mode_candidates if col in df.columns),
    None
)

if mode_col is not None:
    plt.figure(figsize=(10, 5))

    df[mode_col].value_counts().plot(
        kind="bar"
    )

    plt.title("Booking Mode Distribution")
    plt.xlabel("Booking Mode")
    plt.ylabel("Number of Records")
    plt.xticks(rotation=45)
    plt.grid(axis="y")
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / "booking_mode_distribution.png",
        dpi=300
    )

    plt.show()
else:
    print("Booking mode column not found.")

In [ ]:
segment_candidates = ["SEGMENT", "segment", "Segment"]

segment_col = next(
    (col for col in segment_candidates if col in df.columns),
    None
)

if segment_col is not None:
    plt.figure(figsize=(10, 5))

    df[segment_col].value_counts().head(15).plot(
        kind="bar"
    )

    plt.title("Booking Segment Distribution")
    plt.xlabel("Segment")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=45)
    plt.grid(axis="y")
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / "segment_distribution.png",
        dpi=300
    )

    plt.show()
else:
    print("Segment column not found.")

In [ ]:
if len(numeric_columns) >= 2:
    correlation = df[numeric_columns].corr()

    plt.figure(figsize=(12, 8))

    sns.heatmap(
        correlation,
        annot=True,
        cmap="coolwarm",
        fmt=".2f",
        linewidths=0.5
    )

    plt.title("Correlation Between Numeric Features")
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / "correlation_heatmap.png",
        dpi=300
    )

    plt.show()

    display(correlation)
else:
    print("At least two numeric columns are needed.")

In [ ]:
unique_summary = pd.DataFrame({
    "Column": df.columns,
    "Unique_Values": [
        df[col].nunique(dropna=True)
        for col in df.columns
    ]
})

display(
    unique_summary.sort_values(
        by="Unique_Values",
        ascending=False
    )
)

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Duplicate Rows",
        "Total Missing Values",
        "Numeric Columns",
        "Categorical Columns"
    ],
    "Value": [
        len(df),
        len(df.columns),
        int(df.duplicated().sum()),
        int(df.isnull().sum().sum()),
        len(numeric_columns),
        len(categorical_columns)
    ]
})

display(summary)

summary.to_csv(
    PROJECT_DIR / "data" / "processed" /
    "exploration_summary.csv",
    index=False
)

print("Exploration summary saved successfully!")